In [1]:
from pathlib import Path
import importlib.util
import subprocess
import sys

def ensure_package(import_name):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", import_name])
    return __import__(import_name)

IS_COLAB = importlib.util.find_spec("google.colab") is not None

duckdb = ensure_package("duckdb")
pd = ensure_package("pandas")

con = duckdb.connect()

def query_df(query):
    return con.sql(query).df()

if IS_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    dataset_root = Path("/content/drive/MyDrive/Practicum/NGIDS/NGIDS-DS-v1/parquet")
else:
    dataset_root = Path("../dataset")

host_logs = dataset_root / "host_logs.parquet"
ground_truth = dataset_root / "ground_truth.parquet"
syscall_lookup = dataset_root / "syscall-lookup-linux-v3_13.csv"

print(f"Running in Colab: {IS_COLAB}")
print(f"host_logs: {host_logs}")
print(f"ground_truth: {ground_truth}")
print(f"syscall_lookup: {syscall_lookup}")


Running in Colab: False
host_logs: ../dataset/host_logs.parquet
ground_truth: ../dataset/ground_truth.parquet
syscall_lookup: ../dataset/syscall-lookup-linux-v3_13.csv


## Block 1: Dataset Scope

**Analysis Question.** What is the overall scope of the host-log dataset and the anomaly label?


In [2]:
query_df(f"""
SELECT 'host_logs' AS dataset, count(*) AS row_count FROM '{host_logs}'
UNION ALL
SELECT 'ground_truth' AS dataset, count(*) AS row_count FROM '{ground_truth}'
""")


,dataset,row_count
0,host_logs,90054239
1,ground_truth,313926


In [3]:
query_df(f"""
SELECT
    count(*) AS row_count,
    count(*) FILTER (WHERE label = 1) AS anomaly_rows,
    round(100.0 * count(*) FILTER (WHERE label = 1) / count(*), 4) AS anomaly_rate_pct,
    min(date) AS min_date,
    max(date) AS max_date,
    count(DISTINCT path) AS distinct_paths,
    count(DISTINCT pro_id) AS distinct_pro_ids,
    count(DISTINCT sys_call) AS distinct_sys_calls
FROM '{host_logs}'
""")



,row_count,anomaly_rows,anomaly_rate_pct,min_date,max_date,distinct_paths,distinct_pro_ids,distinct_sys_calls
0,90054239,1262427,1.4019,2016-03-11,2016-03-16,100,5576,122


**Short inference.** `host_logs.parquet` has around 90054239 rows, among that 1262427 rows are attack rows. This is simulated for 5 days from 11/03/2016 to 16/03/2016.


In [4]:
query_df(f"DESCRIBE SELECT * FROM '{host_logs}'")


,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,time,TIME,YES,None,None,None
2,pro_id,BIGINT,YES,None,None,None
3,path,VARCHAR,YES,None,None,None
4,sys_call,BIGINT,YES,None,None,None
5,event_id,BIGINT,YES,None,None,None
6,attack_cat,VARCHAR,YES,None,None,None
7,attack_subcat,VARCHAR,YES,None,None,None
8,label,BIGINT,YES,None,None,None


In [93]:
query_df(f"SELECT * FROM '{host_logs}' LIMIT 10")


,date,time,pro_id,path,sys_call,event_id,attack_cat,attack_subcat,label
0,2016-03-11,02:45:01,1830,/sbin/upstart-dbus-bridge,142,45354,normal,normal,0
1,2016-03-11,02:45:06,1804,/bin/dbus-daemon,256,45352,normal,normal,0
2,2016-03-11,02:45:06,2133,/usr/lib/i386-linux-gnu/gconf/gconfd-2,168,45372,normal,normal,0
3,2016-03-11,02:45:35,4528,/usr/bin/python3.4,3,39459,normal,normal,0
4,2016-03-11,02:45:44,1847,/usr/bin/ibus-daemon,102,37263,normal,normal,0
5,2016-03-11,02:45:44,1907,/usr/lib/ibus/ibus-ui-gtk3,168,37896,normal,normal,0
6,2016-03-11,02:45:44,1925,/usr/lib/ibus/ibus-engine-simple,168,37542,normal,normal,0
7,2016-03-11,02:45:44,4461,/usr/sbin/apache2,142,37647,normal,normal,0
8,2016-03-11,02:45:45,1081,/usr/bin/Xorg,102,37480,normal,normal,0
9,2016-03-11,02:45:11,3989,/sbin/auditd,256,45374,normal,normal,0


In [94]:
query_df(f"SELECT * FROM '{host_logs}' where label=1 LIMIT 10")


,date,time,pro_id,path,sys_call,event_id,attack_cat,attack_subcat,label
0,2016-03-11,02:48:03,2110,/usr/lib/libreoffice/program/soffice.bin,102,57713,Exploits,Office Document Batch,1
1,2016-03-11,02:48:03,2110,/usr/lib/libreoffice/program/soffice.bin,102,57763,Exploits,Office Document Batch,1
2,2016-03-11,02:48:03,2110,/usr/lib/libreoffice/program/soffice.bin,265,57659,Exploits,Office Document Batch,1
3,2016-03-11,02:48:03,2110,/usr/lib/libreoffice/program/soffice.bin,265,57707,Exploits,Office Document Batch,1
4,2016-03-11,02:48:03,2110,/usr/lib/libreoffice/program/soffice.bin,265,57749,Exploits,Office Document Batch,1
5,2016-03-11,02:48:03,2110,/usr/lib/libreoffice/program/soffice.bin,3,57701,Exploits,Office Document Batch,1
6,2016-03-11,02:48:03,4493,/usr/lib/libreoffice/program/soffice.bin,256,57740,Exploits,Office Document Batch,1
7,2016-03-11,02:48:03,4493,/usr/lib/libreoffice/program/soffice.bin,78,57686,Exploits,Office Document Batch,1
8,2016-03-11,02:48:03,4493,/usr/lib/libreoffice/program/soffice.bin,78,57694,Exploits,Office Document Batch,1
9,2016-03-11,02:48:03,4493,/usr/lib/libreoffice/program/soffice.bin,78,57718,Exploits,Office Document Batch,1


In [95]:
query_df(f"select * from '{ground_truth}' where attack_cat='Exploits' LIMIT 10")

,date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips
0,2016-03-11,3:07:12,Exploits,Clientside,Oracle Outside In XPM Image Processing Stack O...,CVSS-Critical (https://strikecenter.bpointsys....,TCP 175.45.176.1:13276->10.40.85.32:25
1,2016-03-14,10:33:36,Exploits,Clientside,Oracle Outside In XPM Image Processing Stack O...,CVSS-Critical (https://strikecenter.bpointsys....,TCP 175.45.176.2:19537->10.40.85.32:25
2,2016-03-14,12:43:12,Exploits,Clientside,Oracle Outside In XPM Image Processing Stack O...,CVSS-Critical (https://strikecenter.bpointsys....,TCP 175.45.176.0:7727->10.40.85.32:25
3,2016-03-11,8:10:17,Exploits,Clientside Microsoft Office Batch,Microsoft Office Excel Formula Record PtgExtra...,CVE 2010-3239 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.3:45583->10.40.85.32:25
4,2016-03-11,4:48:33,Exploits,Clientside Microsoft Office Batch,Microsoft Office Excel Formula Record PtgExtra...,CVE 2010-3231 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.1:4714->10.40.85.32:25
5,2016-03-14,9:45:22,Exploits,Clientside Microsoft Office Batch,Microsoft Word ActiveX ScriptBridge Double Fre...,CVE 2010-3331 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.1:27913->10.40.85.32:25
6,2016-03-11,6:08:30,Exploits,Clientside Microsoft Office Batch,Microsoft Office Powerpoint OEPlaceHolderAtom ...,CVE 2010-0032 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.2:13285->10.40.85.32:25
7,2016-03-14,12:14:24,Exploits,Clientside Microsoft Office Batch,Microsoft Office Powerpoint Legacy File Parsin...,CVE 2010-2572 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.2:16441->10.40.85.32:25
8,2016-03-11,7:26:24,Exploits,Clientside Microsoft Office Batch,Microsoft Office Excel Formula Record PtgExtra...,CVE 2010-3239 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.1:2005->10.40.85.32:25
9,2016-03-11,8:09:36,Exploits,Clientside Microsoft Office Batch,Microsoft Office Excel Formula Record PtgExtra...,CVE 2010-3231 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.1:26343->10.40.85.32:25


In [75]:
query_df(f"""
SELECT DISTINCT attack_cat
FROM '{ground_truth}'
WHERE attack_cat NOT LIKE '%->%'
  AND attack_cat NOT LIKE '%:%'
""")

,total_rows,corrupted_rows
0,313926,313425


In [90]:

query_df(f"select * from '{ground_truth}' where attack_cat like '%->%'")

,date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips
0,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:30961->10.40.85.32:80,3:21:36,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
1,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.1:32954->10.40.85.32:80,10:48:00,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
2,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:30310->10.40.85.32:80,7:26:24,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
3,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:46216->10.40.85.32:80,4:33:36,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
4,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.3:12939->10.40.85.32:80,1:26:24,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
...,...,...,...,...,...,...,...
295,2016-03-11,CVE 2009-1535 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:7000->10.40.85.32:80,2:52:48,Exploits,Microsoft IIS Batch,"Microsoft IIS 5.0, IIS 5.1, IIS 6.0 WebDAV Aut..."
296,2016-03-11,CVE 2009-1535 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.0:51081->10.40.85.32:80,12:43:12,Exploits,Microsoft IIS Batch,"Microsoft IIS 5.0, IIS 5.1, IIS 6.0 WebDAV Aut..."
297,2016-03-11,CVE 2009-1535 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:42736->10.40.85.32:80,12:43:12,Exploits,Microsoft IIS Batch,"Microsoft IIS 5.0, IIS 5.1, IIS 6.0 WebDAV Aut..."
298,2016-03-11,CVE 2009-1535 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.0:17101->10.40.85.32:80,7:12:00,Exploits,Microsoft IIS Batch,"Microsoft IIS 5.0, IIS 5.1, IIS 6.0 WebDAV Aut..."


In [82]:
query_df(f"""
SELECT
    COUNT(*) AS total_rows,
    (
        SELECT COUNT(*)
        FROM '{ground_truth}'
        WHERE attack_cat LIKE '%->%'
          AND attack_cat LIKE '%:%'
    ) AS corrupted_rows
FROM '{ground_truth}'
""")

,total_rows,corrupted_rows
0,313926,300


In [22]:
query_df(f"DESCRIBE '{host_logs}'")


,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,time,TIME,YES,None,None,None
2,pro_id,BIGINT,YES,None,None,None
3,path,VARCHAR,YES,None,None,None
4,sys_call,BIGINT,YES,None,None,None
5,event_id,BIGINT,YES,None,None,None
6,attack_cat,VARCHAR,YES,None,None,None
7,attack_subcat,VARCHAR,YES,None,None,None
8,label,BIGINT,YES,None,None,None


In [84]:
query_df(f"DESCRIBE '{ground_truth}'")

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,time,VARCHAR,YES,None,None,None
2,attack_cat,VARCHAR,YES,None,None,None
3,attack_subcat,VARCHAR,YES,None,None,None
4,attack_name,VARCHAR,YES,None,None,None
5,attack_refrence,VARCHAR,YES,None,None,None
6,ips,VARCHAR,YES,None,None,None


In [42]:
query_df(f"""
SELECT pro_id,count(*) AS cnt
FROM '{host_logs}'
WHERE date = '2016-03-11'
  AND time = '3:07:12'
  and label = 1
GROUP BY pro_id
""")

,pro_id,cnt
0,4461,2
1,1853,4
2,1081,96
3,4519,40
4,2110,195
5,4493,40


In [39]:
query_df(f"select * from '{ground_truth}' where date='2016-03-11' and time='3:07:12' and attack_cat='Exploits'")

,date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips
0,2016-03-11,3:07:12,Exploits,Clientside,Oracle Outside In XPM Image Processing Stack O...,CVSS-Critical (https://strikecenter.bpointsys....,TCP 175.45.176.1:13276->10.40.85.32:25
1,2016-03-11,3:07:12,Exploits,Clientside Microsoft Office Batch,Microsoft Office Excel Formula Record PtgExtra...,CVE 2010-3231 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.2:49215->10.40.85.32:25
2,2016-03-11,3:07:12,Exploits,Clientside Microsoft Office Batch,Microsoft Word ActiveX ScriptBridge Double Fre...,CVE 2010-3331 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.1:30084->10.40.85.32:25
3,2016-03-11,3:07:12,Exploits,Clientside Microsoft Office Batch,Microsoft Office Excel Formula Record PtgExtra...,CVE 2010-3239 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.0:56297->10.40.85.32:25
4,2016-03-11,3:07:12,Exploits,Clientside Microsoft Office Batch,Microsoft Office Powerpoint Legacy File Format...,CVE 2011-0976 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.0:31203->10.40.85.32:25
...,...,...,...,...,...,...,...
655,2016-03-11,3:07:12,Exploits,Clientside,Microsoft Powerpoint 2003 Heap Overflow (POP3)...,BPS 2011-0001 (https://strikecenter.bpointsys....,175.45.176.0:27210->10.40.85.32:110
656,2016-03-11,3:07:12,Exploits,Clientside,Windows Media Player ASF Media File Format Par...,CVE 2009-2527 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.0:12987->10.40.85.32:110
657,2016-03-11,3:07:12,Exploits,Clientside,Microsoft Office Text Converter Integer Underf...,CVE 2009-0087 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.1:8467->10.40.85.32:143
658,2016-03-11,3:07:12,Exploits,Clientside,Microsoft PowerPoint Viewer TextChars Atom Rec...,CVE 2010-0034 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.0:6803->10.40.85.32:80


In [28]:
query_df(f"select distinct attack_name from '{ground_truth}' where attack_cat='Exploits'")

,attack_name
0,Adobe Flash MP4 Memory Corruption (https://str...
1,Microsoft Office Component Insecure Library Lo...
2,Microsoft GDI DIBBITBLT HeaderSize Integer Ove...
3,Microsoft DirectShow AVI Invalid biCrlUsed Val...
4,Microsoft Powerpoint 2003 Heap Overflow (SMTP ...
...,...
469,Microsoft GDI+ WMF Integer Overflow (SMTP UUEn...
470,Microsoft Outlook SMB Attachment Vulnerability...
471,Microsoft GDI+ BMP Integer Overflow (SMTP Quot...
472,Wordpad and Windows Shell Com Validation Vulne...


In [41]:
query_df(f"SELECT * FROM '{ground_truth}' WHERE  attack_cat='Exploits' and date='2016-03-11' and time='3:07:12'")

,date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips
0,2016-03-11,3:07:12,Exploits,Clientside,Oracle Outside In XPM Image Processing Stack O...,CVSS-Critical (https://strikecenter.bpointsys....,TCP 175.45.176.1:13276->10.40.85.32:25
1,2016-03-11,3:07:12,Exploits,Clientside Microsoft Office Batch,Microsoft Office Excel Formula Record PtgExtra...,CVE 2010-3231 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.2:49215->10.40.85.32:25
2,2016-03-11,3:07:12,Exploits,Clientside Microsoft Office Batch,Microsoft Word ActiveX ScriptBridge Double Fre...,CVE 2010-3331 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.1:30084->10.40.85.32:25
3,2016-03-11,3:07:12,Exploits,Clientside Microsoft Office Batch,Microsoft Office Excel Formula Record PtgExtra...,CVE 2010-3239 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.0:56297->10.40.85.32:25
4,2016-03-11,3:07:12,Exploits,Clientside Microsoft Office Batch,Microsoft Office Powerpoint Legacy File Format...,CVE 2011-0976 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.0:31203->10.40.85.32:25
...,...,...,...,...,...,...,...
655,2016-03-11,3:07:12,Exploits,Clientside,Microsoft Powerpoint 2003 Heap Overflow (POP3)...,BPS 2011-0001 (https://strikecenter.bpointsys....,175.45.176.0:27210->10.40.85.32:110
656,2016-03-11,3:07:12,Exploits,Clientside,Windows Media Player ASF Media File Format Par...,CVE 2009-2527 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.0:12987->10.40.85.32:110
657,2016-03-11,3:07:12,Exploits,Clientside,Microsoft Office Text Converter Integer Underf...,CVE 2009-0087 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.1:8467->10.40.85.32:143
658,2016-03-11,3:07:12,Exploits,Clientside,Microsoft PowerPoint Viewer TextChars Atom Rec...,CVE 2010-0034 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.0:6803->10.40.85.32:80
